## import libraries

In [102]:
import pandas as pd
from geotessera import GeoTessera
from pathlib import Path
import os

## read datasets

In [103]:
data_root = Path("../_data/exports/crop_country_exports")

In [104]:
def get_inventory(root_path):
    inventory = []

    for file_path in root_path.glob("*.csv"):
        parts = file_path.stem.split('_')
        
        if len(parts) > 3:
            crop = "_".join(parts[:-2])
            country = parts[-2]
            year = int(parts[-1])

        else:  
            crop = parts[0]
            country = parts[-2]
            year = int(parts[-1])

        inventory.append({
            "crop": crop,
            "country": country,
            "year": year,
            "path": file_path
        })

    df = pd.DataFrame(inventory)

    if not df.empty:
        df = df.sort_values(by=["crop", 'year']).reset_index(drop=True)

    return df

In [105]:
available_datasets = get_inventory(data_root)
available_datasets

,crop,country,year,path
0,maize_corn_popcorn,at,2015,../_data/exports/crop_country_exports/maize_co...
1,maize_corn_popcorn,ee,2016,../_data/exports/crop_country_exports/maize_co...
2,maize_corn_popcorn,at,2016,../_data/exports/crop_country_exports/maize_co...
3,maize_corn_popcorn,be,2016,../_data/exports/crop_country_exports/maize_co...
4,maize_corn_popcorn,dk,2016,../_data/exports/crop_country_exports/maize_co...
...,...,...,...,...
88,winter_barley,fr,2021,../_data/exports/crop_country_exports/winter_b...
89,winter_barley,at,2021,../_data/exports/crop_country_exports/winter_b...
90,winter_barley,fr,2022,../_data/exports/crop_country_exports/winter_b...
91,winter_barley,at,2022,../_data/exports/crop_country_exports/winter_b...


## filter dataframe

In [106]:
year = 2023
country = "fr"

In [107]:
def filter_df(df, country, year):
    return df[(df['country'] == country) & (df['year'] == year)]

In [108]:
filtered_dataset = filter_df(available_datasets, country, year)
filtered_dataset

,crop,country,year,path
31,maize_corn_popcorn,fr,2023,../_data/exports/crop_country_exports/maize_co...
63,potatoes,fr,2023,../_data/exports/crop_country_exports/potatoes...
92,winter_barley,fr,2023,../_data/exports/crop_country_exports/winter_b...


In [109]:
def create_dataset(df):
    dataset = {}

    for (crop, country, year), group in df.groupby(['crop', 'country', 'year']):
        key = f"{crop}_{country}_{year}"
        
        dataset[key] = pd.concat(
            [pd.read_csv(p) for p in group['path']],
            ignore_index=True
        )
    
    return dataset

In [110]:
dataset = create_dataset(filtered_dataset)
dataset.keys()

dict_keys(['maize_corn_popcorn_fr_2023', 'potatoes_fr_2023', 'winter_barley_fr_2023'])

In [111]:
dataset['maize_corn_popcorn_fr_2023']

,country_id,year,hcat4_code,hcat4_name,long,lat
0,fr,2023,3310106000,maize_corn_popcorn,1.983122,43.923622
1,fr,2023,3310106000,maize_corn_popcorn,2.822477,49.794996
2,fr,2023,3310106000,maize_corn_popcorn,2.019318,44.799003
3,fr,2023,3310106000,maize_corn_popcorn,-2.018878,48.169992
4,fr,2023,3310106000,maize_corn_popcorn,-1.600655,48.480753
...,...,...,...,...,...,...
495,fr,2023,3310106000,maize_corn_popcorn,4.248254,49.848570
496,fr,2023,3310106000,maize_corn_popcorn,7.775400,48.943218
497,fr,2023,3310106000,maize_corn_popcorn,7.388735,47.945949
498,fr,2023,3310106000,maize_corn_popcorn,2.513489,44.258146


## retrieve embeddings

In [112]:
gt = GeoTessera(embeddings_dir="../_data/embeddings_dir")

In [113]:
def process_embeddings(gdf, country_id, year, crop, include_metadate=True):
    coords = list(zip(gdf['long'], gdf['lat']))

    print(f"Extracting embeddings for {crop}_{country_id}_{year}...")
    embeddings, metadata_list = gt.sample_embeddings_at_points(coords, 
                                                               year=year, 
                                                               include_metadata=include_metadate)
    print(f"Embeddings extracted for {crop}_{country_id}_{year}!")
    return embeddings, metadata_list, coords, country_id

In [114]:
def emb_meta_to_df(embedding, metadata, coords, crop_name, country_id, year):
    print(f"Converting information to a dataframe for {crop_name}_{country_id}_{year}...")
    
    safe_metadata = [m if m is not None else {} for m in metadata]
    df_row = pd.DataFrame(safe_metadata)

    safe_embeddings = []
    for e in embedding:
        if e is not None:
            safe_embeddings.append(e.flatten())
        else:
            safe_embeddings.append(None) 
    
    df_row['embedding'] = safe_embeddings
    df_row['long_lat'] = coords
    df_row['crop'] = crop_name
    df_row['country_id'] = country_id
    df_row['year'] = year

    # df_row = df_row.fillna("-")

    print(f"Information converted for {country_id}! Total rows: {len(df_row)} \n")
    return df_row

# after 
# oi = embeddings2.flatten()
# oi.reshape(1, 128).shape

In [115]:
dataset.keys()

dict_keys(['maize_corn_popcorn_fr_2023', 'potatoes_fr_2023', 'winter_barley_fr_2023'])

In [116]:
dataset['potatoes_fr_2023']['hcat4_name'][0]

'potatoes'

In [117]:
result = []

for key in dataset.keys():
    crop = dataset[key]['hcat4_name'][0]
    gdf = dataset[key]
    embd, meta, coords, country_id = process_embeddings(gdf, country, year, crop)
    result_df = emb_meta_to_df(embd, meta, coords, crop, country_id, year)

    result.append(result_df)

Extracting embeddings for maize_corn_popcorn_fr_2023...
Embeddings extracted for maize_corn_popcorn_fr_2023!
Converting information to a dataframe for maize_corn_popcorn_fr_2023...
Information converted for fr! Total rows: 500 

Extracting embeddings for potatoes_fr_2023...
Embeddings extracted for potatoes_fr_2023!
Converting information to a dataframe for potatoes_fr_2023...
Information converted for fr! Total rows: 500 

Extracting embeddings for winter_barley_fr_2023...
Embeddings extracted for winter_barley_fr_2023!
Converting information to a dataframe for winter_barley_fr_2023...
Information converted for fr! Total rows: 500 



## export data

In [118]:
export_folder = "../_data/exports/embeddings_exports"

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created directory: {export_folder}")

for df in result:
    country = df['country_id'].iloc[0]
    crop = df['crop'].iloc[0]
    year = df['year'].iloc[0]

    file_name = f"{crop}_{country}_{year}_embedding.parquet"
    full_path = os.path.join(export_folder, file_name)
    df.to_parquet(full_path, index=False, engine='pyarrow')
    print(f"Saved: {full_path}")

# To load it back later:
# df = pd.read_parquet('country_crop_embeddings.parquet')

Saved: ../_data/exports/embeddings_exports/maize_corn_popcorn_fr_2023_embedding.parquet
Saved: ../_data/exports/embeddings_exports/potatoes_fr_2023_embedding.parquet
Saved: ../_data/exports/embeddings_exports/winter_barley_fr_2023_embedding.parquet
